In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from google.colab import files
import io

print("Please upload 5 files: NO2, CO, Wind_U, Wind_V, and TEMP CSVs.")
uploaded_files = files.upload()

dataframes = {}
for name, content in uploaded_files.items():
    df_temp = pd.read_csv(io.BytesIO(content))
    if '.geo' in df_temp.columns: df_temp = df_temp.drop('.geo', axis=1)
    df_temp['timestamp'] = pd.to_datetime(df_temp['timestamp'])
    df_temp['date'] = df_temp['timestamp'].dt.date
    dataframes[name] = df_temp
    print(f"Loaded {name}")

In [ ]:
def create_grid_tensor(dfs, feature_list, lat_res=0.01, lon_res=0.01):
    """
    Creates a standardized tensor for a list of features across a common date range.
    """
    # Determine common date range
    all_dates = pd.to_datetime(pd.concat([df['date'] for df in dfs.values()]).unique()).sort_values()

    # For simplicity in this workflow, we align grids by rounding
    # In a real scenario, you would use scipy.interpolate.griddata
    combined_data = {}
    for feat in feature_list:
        # Find the DF that contains this feature
        target_df = next(df for name, df in dfs.items() if feat.upper() in name.upper())
        combined_data[feat] = target_df

    print(f"Standardizing features: {feature_list}")
    # Logic to return tensor [Days, Channels, Lat, Lon] goes here
    return all_dates

In [ ]:
# ### Data Tensor Preparation
# We prepare two sets of tensors:
# 1. **Pollutants**: (NO2, CO)
# 2. **Meteorology**: (Wind_U, Wind_V, TEMP)

# These will be passed through the multi-branch CNN.

### CNN Data Preparation
Defining spatial and temporal radii for interpolation and rounding coordinates to 3 decimal places for a high-resolution grid.

In [ ]:
# Placeholder for Tensor Generation (NO2 and TEMP only)
batch_size = 30
# pol_in: [Batch, 1 (NO2), Lat, Lon]
pol_in = torch.randn(batch_size, 1, 32, 32)
# met_in: [Batch, 1 (TEMP), Lat, Lon]
met_in = torch.randn(batch_size, 1, 32, 32)

print(f"Pollutant Tensor Shape (NO2): {pol_in.shape}")
print(f"Meteo Tensor Shape (TEMP): {met_in.shape}")

### Torch Tensor Structuring
We will now prepare a function to interpolate missing values within the defined radii and organize the result into a PyTorch tensor of shape `(Date, Lat, Lon, Features)`.

In [ ]:
# 1. Mapping feature names dynamically to uploaded file names
feature_map = {
    'NO2': [k for k in dataframes.keys() if 'NO2' in k][0],
    'TEMP': [k for k in dataframes.keys() if 'TEMP' in k][0]
}

# 2. Define coordinate grids from the NO2 data (reference grid)
ref_df = dataframes[feature_map['NO2']]
unique_dates = np.sort(ref_df['date'].unique())
unique_lats = np.sort(ref_df['latitude'].round(3).unique())
unique_lons = np.sort(ref_df['longitude'].round(3).unique())

# 3. Define the interpolation helper
def interpolate_feature(target_date, target_lat, target_lon, df, val_col, s_rad=0.02, t_rad=1):
    t_mask = (df['date'] >= target_date - pd.Timedelta(days=t_rad)) & (df['date'] <= target_date + pd.Timedelta(days=t_rad))
    s_mask = (np.sqrt((df['latitude'] - target_lat)**2 + (df['longitude'] - target_lon)**2) <= s_rad)
    subset = df[t_mask & s_mask & df[val_col].notna()]
    return subset[val_col].mean() if not subset.empty else 0.0

# 4. Set parameters for the tensor
s_radius_deg = 0.02
t_radius_days = 1
features = ['NO2', 'TEMP']

print(f"Grids defined: {len(unique_dates)} dates, {len(unique_lats)} lats, {len(unique_lons)} lons.")

In [ ]:
# Initialize the tensor: [Days, Lats, Lons, Channels]
n_days, n_lats, n_lons, n_feats = len(unique_dates), len(unique_lats), len(unique_lons), len(features)
data_tensor = np.zeros((n_days, n_lats, n_lons, n_feats))

print(f"Populating tensor of shape: {data_tensor.shape}...")

for d_idx, t_date in enumerate(unique_dates):
    for lat_idx, t_lat in enumerate(unique_lats):
        for lon_idx, t_lon in enumerate(unique_lons):
            for f_idx, feat_name in enumerate(features):
                source_df = dataframes[feature_map[feat_name]]
                # Using 'feat_name' as the value column key (NO2 or TEMP)
                val = interpolate_feature(t_date, t_lat, t_lon, source_df, feat_name, s_radius_deg, t_radius_days)
                data_tensor[d_idx, lat_idx, lon_idx, f_idx] = val

# Convert to PyTorch tensor and handle NaNs if necessary (e.g., zero fill)
torch_tensor = torch.from_numpy(data_tensor).float()
torch_tensor = torch.nan_to_num(torch_tensor, nan=0.0)

print("Tensor generation complete.")
print(f"Final Tensor Shape: {torch_tensor.shape}")

### Multi-Branch CNN with Attention
This model processes pollutants and meteorological data through separate convolutional branches, fuses them via a simplified attention mechanism, and upscales the result to a single-day prediction.

In [ ]:
class AttentionFusion(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.query = nn.Linear(dim, dim)
        self.key = nn.Linear(dim, dim)
        self.value = nn.Linear(dim, dim)

    def forward(self, x):
        b, c, h, w = x.shape
        flat_x = x.view(b, c, -1).permute(0, 2, 1)
        q, k, v = self.query(flat_x), self.key(flat_x), self.value(flat_x)
        attn = F.softmax(torch.bmm(q, k.transpose(1, 2)) / (c**0.5), dim=-1)
        out = torch.bmm(attn, v).permute(0, 2, 1).view(b, c, h, w)
        return out

class MultiScaleModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Branch 1: Pollutants (Only NO2)
        self.branch_pol = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU())
        # Branch 2: Meteo (Only TEMP)
        self.branch_met = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU())

        self.attention = AttentionFusion(32)
        self.upscale = nn.Sequential(
            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),

            nn.ReLU(),
            nn.Conv2d(16, 2, 3, padding=1) # Output 2 features (NO2, TEMP) for 1 day
        )

    def forward(self, pol_tensor, met_tensor):
        p = self.branch_pol(pol_tensor)
        m = self.branch_met(met_tensor)
        combined = torch.cat([p, m], dim=1)
        fused = self.attention(combined)
        return self.upscale(fused)

### Model Instantiation and Forward Pass Demonstration
Now, let's instantiate our `SimpleCNN` model and demonstrate its forward pass using the prepared `torch_tensor` data.

In [ ]:
model = MultiScaleModel()

# Forward pass
# The model takes two inputs and predicts 5 feature maps for a single target day
# Here batch_size represents different sequences or days
output = model(pol_in, met_in)

print(f"Output Shape: {output.shape} (Batch, 5 Features, Lat_Out, Lon_Out)")

### Model Training and Evaluation
We will now train the `SimpleCNN` using the prepared `torch_tensor` as both the input and the target (reconstruction task) to demonstrate the training loop. We will use the Mean Squared Error (MSE) loss and the Adam optimizer.

In [ ]:
# Training logic for Multi-Scale Model (2 output features)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Target: 2 features (NO2, TEMP) upscaled
target = torch.randn(batch_size, 2, 64, 64)

output = model(pol_in, met_in)
loss = criterion(output, target)
loss.backward()
optimizer.step()
print(f"Initial Loss (NO2 + TEMP): {loss.item()}")

### Testing on the same data
Finally, we evaluate the model performance on the training data to see the final reconstruction loss.

In [ ]:
model.eval()
with torch.no_grad():
    # Reshape torch_tensor to match multi-branch expected input
    # Original: [Days, Lats, Lons, 2] -> Need [Days, 1, Lats, Lons] for each branch
    pol_test = torch_tensor[:, :, :, 0].unsqueeze(1)
    met_test = torch_tensor[:, :, :, 1].unsqueeze(1)

    final_output = model(pol_test, met_test)
    # Creating a dummy target for loss calculation demonstration
    dummy_target = torch.randn(final_output.shape)
    test_loss = criterion(final_output, dummy_target)

print(f"Final Inference Shape: {final_output.shape}")
print(f"Test Loss: {test_loss.item():.6f}")